# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step template for loading and exploring a dataset with multiple record sets and fields using the `mlcroissant` library.

### Dataset Source
The dataset is described by a Croissant schema at:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure mlcroissant is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load the dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Get metadata
meta = dataset.metadata

print(f"Dataset Name: {meta.name}\n")
print(f"Description: {meta.description}\n")
print(f"Published: {meta.date_published}")

## 2. Data Overview
List all available record sets, fields, and columns. **All entities are referenced by their `@id`.**

Let's inspect what record sets are available, then for each, examine the fields and columns (by their `@id`).

In [ ]:
# List all record sets in the dataset
record_sets = list(meta.record_sets)
if not record_sets:
    print("No RecordSets found in metadata. Using datafile(s) directly, if available.")
    # For datasets where record_sets is empty, use each distribution as one record set
    record_sets = [dist['@id'] for dist in meta.to_json().get('distribution', [])]
    print(f"Distribution-based RecordSets: {record_sets}")
else:
    print("RecordSets with @id and name:")
    for rs in record_sets:
        print(f"  @id: {rs['@id']}, name: {getattr(rs,'name',None)}")

# For each record set, list out all field @ids and columns
from pprint import pprint
for rs_id in record_sets:
    print("\n---")
    print(f"[RecordSet] @id: {rs_id}")
    # Try to get the fields, if it exists
    try:
        if isinstance(rs_id, dict):
            rs_id_val = rs_id['@id']
        else:
            rs_id_val = rs_id
        rs = dataset.metadata.record_set(rs_id_val)
        print(f"  Name: {getattr(rs,'name',None)}")
        print("  Fields (@id):")
        for f in getattr(rs, 'fields', []):
            print(f"    {f['@id']}")
        print("  Columns (@id):")
        for c in getattr(rs, 'columns', []):
            print(f"    {c['@id']}")
    except Exception as e:
        print(f"  Could not load fields/columns for {rs_id}: {e}")

## 3. Data Extraction
Load each available record set (by `@id`) into a pandas DataFrame.

> **Note**: For this dataset, if there are no explicit Croissant `RecordSet`s, we directly use the main datafile's `@id` as the record set.

In [ ]:
# Prepare the list of record set @ids (from the previous cell, using distribution as fallback)
record_set_ids = [rs['@id'] if isinstance(rs, dict) and '@id' in rs else rs for rs in record_sets]
print(f"RecordSet @ids to extract: {record_set_ids}")

dataframes = {}
for rs_id in record_set_ids:
    print(f"\nLoading records for RecordSet @id: {rs_id}")
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded dataframe with shape: {df.shape}")
        print(f"Columns: {df.columns.tolist()}")
    except Exception as e:
        print(f"  Could not load records for {rs_id}: {e}")

# Pick the first loaded record set with data for quick look
main_rs_id = None
for k,v in dataframes.items():
    if v.shape[0] > 0:
        main_rs_id = k
        break
if main_rs_id:
    print(f"\nFirst rows of RecordSet {main_rs_id}:")
    display(dataframes[main_rs_id].head())
else:
    print("No loaded record sets with data.")

## 4. Exploratory Data Analysis (EDA)
Let's select a numeric field (by its column `@id`) and perform basic operations, such as filtering, normalizing, and grouping.

**You may need to refer to the list of columns from the prior cells to pick relevant numeric fields.**

In [ ]:
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Pick a record set with data
rs_id = main_rs_id
df = dataframes[rs_id]

print(f"Data shape: {df.shape}; Columns: {df.columns.tolist()}")

# Attempt to auto-choose a numeric column
numeric_col_candidates = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
if not numeric_col_candidates:
    # Try to coerce columns to numeric and see which ones succeed for EDA
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col], errors='coerce')
        except Exception:
            continue
    numeric_col_candidates = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]

print(f"Numeric candidate fields: {numeric_col_candidates}")

# We'll use the first numeric column found
if numeric_col_candidates:
    numeric_field_id = numeric_col_candidates[0]
    threshold = df[numeric_field_id].mean() if not np.isnan(df[numeric_field_id].mean()) else 0
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"\nFiltered records with {numeric_field_id} > {threshold:.2f} (mean value):")
    display(filtered_df.head())

    # Normalize
    col_norm = f"{numeric_field_id}_normalized"
    filtered_df[col_norm] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std(ddof=0)
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, col_norm]].head())

    # Try to pick a grouping field (categorical with low cardinality & not the numeric field)
    cat_candidates = [col for col in df.columns if df[col].dtype==object and df[col].nunique() < min(10, max(4, len(df)//10)) and col!=numeric_field_id]
    group_field = cat_candidates[0] if cat_candidates else None
    print(f"Grouping candidate field: {group_field}")
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
        print(f"\nMean {numeric_field_id} by {group_field}:")
        display(grouped_df.head())
else:
    print("No numeric field found in the data for EDA.")

## 5. Visualization
Let's visualize the filtered and normalized numeric field distribution and optionally, a barplot of grouped mean values.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_col_candidates:
    plt.figure(figsize=(8,4))
    sns.histplot(filtered_df[numeric_field_id].dropna(), kde=True, bins=20)
    plt.title(f"Distribution of {numeric_field_id} (filtered)")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

    # Normalized values
    plt.figure(figsize=(8,4))
    sns.histplot(filtered_df[col_norm].dropna(), kde=True, bins=20)
    plt.title(f"Normalized {numeric_field_id} (filtered)")
    plt.xlabel(col_norm)
    plt.ylabel('Frequency')
    plt.show()

    # Grouped mean plot, if group_field found
    if group_field is not None:
        plt.figure(figsize=(8,4))
        sns.barplot(x=group_field, y=numeric_field_id, data=grouped_df)
        plt.title(f"Mean {numeric_field_id} by {group_field}")
        plt.show()
else:
    print("No numeric field to visualize.")

## 6. Conclusion
In this notebook we loaded a FAIR^2-compliant protein mass spectrometry dataset with Croissant metadata using the `mlcroissant` library. We explored the available record sets and columns, loaded records by referencing their `@id`, and performed basic EDA and visualization on the numeric fields.

This workflow can be extended for downstream proteomics analysis, statistical comparisons, and more advanced visualizations.

**Tip:** Always reference record sets, fields, and columns by their `@id` as required by the Croissant schema to ensure robust and reproducible workflows.